In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os 

In [11]:
load_dotenv()
google_api_key = os.getenv('GOOGLE_API_KEY')

In [12]:
model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    api_key=google_api_key,  # Changed from google_api_key to api_key
    max_retries=0,
)

In [13]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [14]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [15]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [16]:
def evaluate_blog(state: BlogState) -> BlogState:

    content = state['content']

    prompt = f'Evaluate the following blog content for its quality and coherence. Provide a score out of 10 and a brief explanation for the score.\n\n{content}'

    evaluation = model.invoke(prompt).content

    state['evaluation'] = evaluation

    return state

In [17]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)  

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()


In [18]:
intial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': '## Detailed Blog Outline: The Rise of AI in India: Transforming Industries and Shaping the Future\n\n**Blog Title Options:**\n\n* **The AI Awakening: How Artificial Intelligence is Reshaping India\'s Landscape**\n* **From Silicon Valley to Silicon Valley of the East: Decoding the Rise of AI in India**\n* **India\'s AI Revolution: Opportunities, Challenges, and the Road Ahead**\n* **Powering Progress: Examining the Explosive Growth of Artificial Intelligence in India**\n\n---\n\n### I. Introduction (Setting the Stage)\n\n**A. Engaging Hook:**\n    * Start with a compelling statistic about India\'s digital penetration or a recent high-profile AI success story in the country (e.g., AI in healthcare diagnostics, government initiatives).\n    * Acknowledge India\'s historical strength in software and IT, positioning AI as the next major technological leap.\n\n**B. Defining AI in the Indian Context:**\n    * Briefly define what AI means in practic

In [19]:
print(final_state['outline'])

## Detailed Blog Outline: The Rise of AI in India: Transforming Industries and Shaping the Future

**Blog Title Options:**

* **The AI Awakening: How Artificial Intelligence is Reshaping India's Landscape**
* **From Silicon Valley to Silicon Valley of the East: Decoding the Rise of AI in India**
* **India's AI Revolution: Opportunities, Challenges, and the Road Ahead**
* **Powering Progress: Examining the Explosive Growth of Artificial Intelligence in India**

---

### I. Introduction (Setting the Stage)

**A. Engaging Hook:**
    * Start with a compelling statistic about India's digital penetration or a recent high-profile AI success story in the country (e.g., AI in healthcare diagnostics, government initiatives).
    * Acknowledge India's historical strength in software and IT, positioning AI as the next major technological leap.

**B. Defining AI in the Indian Context:**
    * Briefly define what AI means in practical terms for India (not just abstract concepts, but real-world appl

In [20]:
print(final_state['content'])

# The AI Awakening: How Artificial Intelligence is Reshaping India's Landscape

India, long recognized as the world’s back office for IT services, is now aggressively positioning itself at the forefront of the next technological revolution: Artificial Intelligence. With digital penetration skyrocketing—evidenced by the staggering volume of daily transactions on the Unified Payments Interface (UPI)—and a demographic dividend ready to embrace disruption, the nation is experiencing an **AI Awakening**. This isn't merely about importing foreign technology; it’s about building bespoke, scalable AI solutions tailored to solve India’s unique, complex challenges.

This blog explores the rapid ascent of AI in India, analyzing its transformative impact across vital sectors, dissecting the foundational elements driving this growth, addressing the critical challenges ahead, and projecting its future trajectory in nation-building.

---

## II. The Foundation: Why India is Ripe for an AI Boom

The c